In [ ]:
pip install merlin-vlm rich pandas nibabel matplotlib numpy torch

In [ ]:
import os, json, warnings
import numpy as np
import nibabel as nib
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd
from rich.console import Console
from rich.table import Table

warnings.filterwarnings("ignore")

PROJECT_ROOT = os.path.expanduser("/home/chest_ct/code")
DATA_ROOT    = os.path.join(PROJECT_ROOT, "data")
VOLUMES_DIR  = os.path.join(DATA_ROOT, "segmentations/segmentations")
REXCT_DIR    = os.path.join(DATA_ROOT, "rexgrounding-ct")
DATASET_JSON = os.path.join(REXCT_DIR, "dataset.json")
RESULTS_DIR  = os.path.join(PROJECT_ROOT, "models/merlin/results_full")
PHENOTYPES_CSV = os.path.join(PROJECT_ROOT, "models/merlin/phenotypes.csv")  # from Merlin repo

os.makedirs(RESULTS_DIR, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU    : {torch.cuda.get_device_name(0)}")

In [ ]:
# Index all .nii.gz files
ct_index = {}
for root, dirs, files in os.walk(VOLUMES_DIR):
    for f in files:
        if f.endswith(".nii.gz"):
            ct_index[f] = os.path.join(root, f)
print(f"CT files on disk: {len(ct_index)}")

# Load dataset JSON
with open(DATASET_JSON) as f:
    metadata = json.load(f)
samples = metadata if isinstance(metadata, list) else list(metadata.values())[0]

# Match JSON to files on disk
matched = [s for s in samples if s["name"] in ct_index]
print(f"Matched: {len(matched)} / {len(samples)}")

In [ ]:
from merlin import Merlin

print("Loading Merlin models...")

model_embed   = Merlin(ImageEmbedding=True).to(DEVICE).eval()
model_pheno   = Merlin(PhenotypeCls=True).to(DEVICE).eval()
model_fiveyear= Merlin(FiveYearPred=True).to(DEVICE).eval()
model_report  = Merlin(RadiologyReport=True).to(DEVICE).eval()

print("All 4 Merlin models loaded!")

# Disease names for five-year prediction
DISEASE_NAMES = [
    "Cardiovascular Disease (CVD)",
    "Ischemic Heart Disease (IHD)",
    "Hypertension (HTN)",
    "Diabetes Mellitus (DM)",
    "Chronic Kidney Disease (CKD)",
    "Chronic Liver Disease (CLD)",
]

# Load phenotype labels
if os.path.exists(PHENOTYPES_CSV):
    phenotypes_df = pd.read_csv(PHENOTYPES_CSV)
else:
    phenotypes_df = None
    print("Warning: phenotypes.csv not found — phenotype labels will show as indices")

In [ ]:
from merlin.data import DataLoader

def make_dataloader(nii_path, cache_dir):
    """Wrap one CT path into Merlin's DataLoader format."""
    datalist = [{"image": nii_path, "text": ""}]
    return DataLoader(
        datalist    = datalist,
        cache_dir   = cache_dir,
        batchsize   = 1,
        shuffle     = False,
        num_workers = 0,
    )

In [ ]:
def compute_gradcam(model, ct_tensor, target_layer_name="layer4"):
    """
    Compute GradCAM heatmap using Merlin's image encoder.
    Returns a 3D numpy heatmap same shape as CT volume.
    """
    activations = {}
    gradients   = {}

    # Find the target layer inside Merlin's image encoder
    target_layer = None
    for name, module in model.named_modules():
        if target_layer_name in name:
            target_layer = module

    if target_layer is None:
        # Fallback: use activation-based heatmap
        return None

    def forward_hook(module, input, output):
        activations["value"] = output.detach()

    def backward_hook(module, grad_in, grad_out):
        gradients["value"] = grad_out[0].detach()

    fh = target_layer.register_forward_hook(forward_hook)
    bh = target_layer.register_full_backward_hook(backward_hook)

    ct_tensor.requires_grad_(True)
    output = model(ct_tensor)
    if isinstance(output, (list, tuple)):
        output = output[0]

    score = output.mean()
    model.zero_grad()
    score.backward()

    fh.remove()
    bh.remove()

    if "value" not in activations or "value" not in gradients:
        return None

    act  = activations["value"].squeeze().cpu().numpy()  # (C, H, W, D) or (C, H, W)
    grad = gradients["value"].squeeze().cpu().numpy()

    weights = grad.mean(axis=tuple(range(1, grad.ndim)))  # mean over spatial dims
    cam     = np.zeros(act.shape[1:], dtype=np.float32)
    for i, w in enumerate(weights):
        cam += w * act[i]

    cam = np.maximum(cam, 0)
    if cam.max() > 0:
        cam = cam / cam.max()

    return cam


def activation_heatmap_fallback(volume_norm, emb_np):
    """Simple activation heatmap when GradCAM isn't available."""
    H, W, D    = volume_norm.shape
    scalar     = float(np.mean(np.abs(emb_np.flatten())))
    heatmap    = np.zeros((H, W, D), dtype=np.float32)
    for d in range(D):
        heatmap[:, :, d] = volume_norm[:, :, d] * scalar
    heatmap = (heatmap - heatmap.min()) / (heatmap.max() - heatmap.min() + 1e-8)
    return heatmap

In [ ]:
def save_visualization(name, ct_vol, heatmap, binary_mask, out_dir):
    """Save a rich PNG showing CT + heatmap overlay + binary mask."""
    ct_display = np.clip(ct_vol, -1000, 400)
    ct_display = (ct_display + 1000) / 1400.0
    H, W, D    = ct_vol.shape

    # Resize heatmap to match CT if needed
    if heatmap is not None and heatmap.shape != ct_vol.shape:
        from scipy.ndimage import zoom
        factors = (H/heatmap.shape[0], W/heatmap.shape[1],
                   D/heatmap.shape[2] if heatmap.ndim == 3 else 1)
        if heatmap.ndim == 2:
            heatmap = np.stack([heatmap]*D, axis=-1)
        heatmap = zoom(heatmap, factors, order=1)

    # Pick 6 slices with most mask activity
    active_slices = [d for d in range(D) if binary_mask[:, :, d].sum() > 0]
    if not active_slices:
        active_slices = list(range(0, D, max(1, D // 6)))
    step   = max(1, len(active_slices) // 6)
    slices = active_slices[::step][:6]

    fig, axes = plt.subplots(3, 6, figsize=(22, 11))
    fig.patch.set_facecolor("#0a0a0a")
    fig.suptitle(f"Merlin Analysis: {name}", color="white", fontsize=13,
                 fontweight="bold", y=0.99)

    row_labels = ["CT scan", "Heatmap overlay", "Binary mask"]

    for col, d in enumerate(slices):
        ct_sl   = ct_display[:, :, d].T
        mask_sl = binary_mask[:, :, d].T
        heat_sl = heatmap[:, :, d].T if heatmap is not None else ct_sl

        # Row 0: raw CT
        axes[0, col].imshow(ct_sl, cmap="gray", origin="lower", vmin=0, vmax=1)
        axes[0, col].set_title(f"z={d}", color="white", fontsize=8)

        # Row 1: CT + heatmap overlay
        axes[1, col].imshow(ct_sl, cmap="gray", origin="lower", vmin=0, vmax=1)
        axes[1, col].imshow(heat_sl, cmap="jet", origin="lower", alpha=0.4,
                            vmin=0, vmax=1)

        # Row 2: binary mask
        axes[2, col].imshow(ct_sl, cmap="gray", origin="lower", vmin=0, vmax=1)
        masked = np.ma.masked_where(mask_sl == 0, mask_sl)
        axes[2, col].imshow(masked, cmap="Reds", origin="lower",
                            alpha=0.6, vmin=0, vmax=1)

        for row in range(3):
            axes[row, col].axis("off")

    # Row labels on left
    for row, label in enumerate(row_labels):
        axes[row, 0].set_ylabel(label, color="white", fontsize=9,
                                rotation=90, labelpad=5)
        axes[row, 0].yaxis.set_label_position("left")
        axes[row, 0].axis("off")

    # Legend
    patches = [
        mpatches.Patch(color="gray",  label="CT"),
        mpatches.Patch(color="red",   alpha=0.5, label="Localization"),
        mpatches.Patch(color="blue",  alpha=0.5, label="High activation"),
    ]
    fig.legend(handles=patches, loc="lower center", ncol=3,
               facecolor="#1a1a1a", edgecolor="white",
               labelcolor="white", fontsize=9,
               bbox_to_anchor=(0.5, 0.005))

    plt.tight_layout(rect=[0, 0.04, 1, 0.97])
    out_path = os.path.join(out_dir, "ct_analysis.png")
    plt.savefig(out_path, dpi=150, bbox_inches="tight",
                facecolor=fig.get_facecolor())
    plt.close()
    return out_path

In [ ]:
def process_volume(sample, models, ct_index, phenotypes_df, out_dir=RESULTS_DIR):
    name     = sample["name"]
    findings = sample["findings"]
    shape    = sample["shape"]
    cats     = sample.get("categories", {})
    pixels   = sample.get("pixels", {})
    protocol = sample.get("protocol", "unknown")

    stem = name.replace(".nii.gz", "")
    out  = os.path.join(out_dir, stem)

    if os.path.exists(os.path.join(out, "findings.json")):
        return "skipped"

    os.makedirs(out, exist_ok=True)
    console = Console()

    try:
        ct_path   = ct_index[name]
        cache_dir = os.path.join(out, "cache")
        os.makedirs(cache_dir, exist_ok=True)

        console.print(f"\n[bold cyan]{'='*55}[/bold cyan]")
        console.print(f"[bold white]Processing:[/bold white] {name}")
        console.print(f"[bold cyan]{'='*55}[/bold cyan]")

        # ── Load raw CT for visualization ─────────────────────
        ct_raw    = nib.load(ct_path)
        ct_vol    = ct_raw.get_fdata()
        affine    = ct_raw.affine
        ct_norm   = np.clip(ct_vol, -1000, 400)
        ct_norm   = (ct_norm + 1000) / 1400.0
        ct_norm   = np.nan_to_num(ct_norm, nan=0.0)
        ct_tensor = torch.tensor(ct_norm, dtype=torch.float32)\
                        .unsqueeze(0).unsqueeze(0).to(DEVICE)

        # ── 1. IMAGE EMBEDDING ────────────────────────────────
        console.print("[yellow]► Getting image embedding...[/yellow]")
        with torch.no_grad():
            emb_out = models["embed"](ct_tensor)
        emb_np = emb_out[0].cpu().numpy() if isinstance(emb_out, (list,tuple)) \
                 else emb_out.cpu().numpy()
        console.print(f"  Embedding shape: {emb_np.shape}  "
                      f"norm: {np.linalg.norm(emb_np):.4f}")

        # ── 2. PHENOTYPE CLASSIFICATION ───────────────────────
        console.print("[yellow]► Phenotype classification...[/yellow]")
        with torch.no_grad():
            pheno_out = models["pheno"](ct_tensor)
        pheno_probs = pheno_out.squeeze(0).cpu().numpy() \
                      if not isinstance(pheno_out, (list,tuple)) \
                      else pheno_out[0].squeeze(0).cpu().numpy()

        top_idx   = np.argsort(pheno_probs)[::-1][:5]
        top_probs = pheno_probs[top_idx]

        table = Table(show_header=True, header_style="bold magenta")
        table.add_column("Phecode",     style="cyan", width=10)
        table.add_column("Description",style="green", max_width=35)
        table.add_column("Probability", style="yellow")
        phenotype_results = []
        for idx, prob in zip(top_idx, top_probs):
            if phenotypes_df is not None:
                row  = phenotypes_df.iloc[idx].values
                code = str(row[0])
                desc = str(row[1])
            else:
                code = str(idx)
                desc = f"Phenotype_{idx}"
            table.add_row(code, desc, f"{prob:.4f}")
            phenotype_results.append({"code": code, "description": desc,
                                      "probability": float(prob)})
        console.print(table)

        # ── 3. FIVE-YEAR DISEASE PREDICTION ──────────────────
        console.print("[yellow]► Five-year disease prediction...[/yellow]")
        with torch.no_grad():
            fiveyear_out = models["fiveyear"](ct_tensor)
        fiveyear_probs = fiveyear_out.squeeze(0).cpu().numpy() \
                         if not isinstance(fiveyear_out, (list,tuple)) \
                         else fiveyear_out[0].squeeze(0).cpu().numpy()

        table2 = Table(show_header=True, header_style="bold blue")
        table2.add_column("Disease",     style="cyan", width=30)
        table2.add_column("Probability", style="yellow", justify="right")
        fiveyear_results = []
        for disease, prob in zip(DISEASE_NAMES, fiveyear_probs):
            table2.add_row(disease, f"{float(prob):.4f}")
            fiveyear_results.append({"disease": disease,
                                     "probability": float(prob)})
        console.print(table2)

        # ── 4. RADIOLOGY REPORT GENERATION ───────────────────
        console.print("[yellow]► Generating radiology report...[/yellow]")
        dataloader = make_dataloader(ct_path, cache_dir)
        generated_report = ""
        with torch.no_grad():
            for batch in dataloader:
                report_out = models["report"](batch["image"].to(DEVICE))
                if isinstance(report_out, str):
                    generated_report = report_out
                elif isinstance(report_out, (list, tuple)):
                    generated_report = report_out[0] if report_out else ""
                else:
                    generated_report = str(report_out)
                break
        console.print(f"  Report: {generated_report[:120]}...")

        # ── 5. LOCALIZATION (GradCAM + binary mask) ───────────
        console.print("[yellow]► Computing localization heatmap...[/yellow]")
        heatmap = compute_gradcam(models["embed"], ct_tensor.clone())
        if heatmap is None:
            heatmap = activation_heatmap_fallback(ct_norm, emb_np)
            console.print("  Used activation fallback heatmap")
        else:
            console.print("  GradCAM computed")

        # Resize heatmap to CT volume shape if needed
        if heatmap.shape != ct_vol.shape:
            from scipy.ndimage import zoom
            if heatmap.ndim == 2:
                heatmap = np.stack([heatmap]*ct_vol.shape[2], axis=-1)
            factors = tuple(ct_vol.shape[i]/heatmap.shape[i] for i in range(3))
            from scipy.ndimage import zoom
            heatmap = zoom(heatmap, factors, order=1)

        threshold   = np.percentile(heatmap, 95)
        binary_mask = (heatmap >= threshold).astype(np.uint8)
        console.print(f"  Active voxels: {binary_mask.sum()} "
                      f"({100*binary_mask.sum()/ct_vol.size:.2f}%)")

        # Free GPU memory
        del ct_tensor
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()

        # ── BUILD FULL REPORT TEXT ─────────────────────────────
        finding_lines = "\n".join(
            f"  [{idx}] {text}\n      Category: {cats.get(idx,'N/A')}  "
            f"Pixels: {pixels.get(idx,'N/A')}"
            for idx, text in findings.items()
        )
        pheno_lines   = "\n".join(
            f"  {r['code']:10s}  {r['description'][:35]:35s}  {r['probability']:.4f}"
            for r in phenotype_results
        )
        disease_lines = "\n".join(
            f"  {r['disease']:35s}  {r['probability']:.4f}"
            for r in fiveyear_results
        )

        full_report = f"""
MERLIN CT ANALYSIS REPORT
{'='*60}
File       : {name}
Protocol   : {protocol}
Shape      : {shape[0]} x {shape[1]} x {shape[2]} voxels

━━━ 1. IMAGE EMBEDDING ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Shape : {emb_np.shape}
  Norm  : {np.linalg.norm(emb_np):.4f}
  Mean  : {np.mean(emb_np):.4f}
  Std   : {np.std(emb_np):.4f}

━━━ 2. DATASET FINDINGS ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
{finding_lines}

━━━ 3. PHENOTYPE CLASSIFICATION (Top 5) ━━━━━━━━━━━━━━━━━━━
  Code        Description                          Probability
{pheno_lines}

━━━ 4. FIVE-YEAR DISEASE PREDICTION ━━━━━━━━━━━━━━━━━━━━━━━
  Disease                                Probability
{disease_lines}

━━━ 5. GENERATED RADIOLOGY REPORT ━━━━━━━━━━━━━━━━━━━━━━━━━
{generated_report}

━━━ 6. LOCALIZATION MASK ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Active voxels : {binary_mask.sum()}
  Threshold     : {threshold:.4f} (top 5% activation)
  Coverage      : {100*binary_mask.sum()/ct_vol.size:.2f}% of volume
{'='*60}
"""
        console.print(full_report)

        # ── SAVE ALL OUTPUTS ───────────────────────────────────

        # 1. Full report
        with open(os.path.join(out, "report.txt"), "w") as f:
            f.write(full_report)

        # 2. Embedding
        np.savez_compressed(os.path.join(out, "embedding.npz"),
                            embedding=emb_np)

        # 3. Localization mask (.nii.gz)
        nib.save(nib.Nifti1Image(binary_mask, affine),
                 os.path.join(out, "localization_mask.nii.gz"))

        # 4. Heatmap (.nii.gz)
        nib.save(nib.Nifti1Image(heatmap.astype(np.float32), affine),
                 os.path.join(out, "heatmap.nii.gz"))

        # 5. Findings JSON (all structured outputs)
        findings_out = {
            "name"                 : name,
            "protocol"             : protocol,
            "shape"                : shape,
            "dataset_findings"     : findings,
            "categories"           : cats,
            "pixels"               : pixels,
            "embedding_norm"       : float(np.linalg.norm(emb_np)),
            "phenotype_top5"       : phenotype_results,
            "fiveyear_predictions" : fiveyear_results,
            "generated_report"     : generated_report,
            "mask_active_voxels"   : int(binary_mask.sum()),
            "mask_coverage_pct"    : float(100*binary_mask.sum()/ct_vol.size),
        }
        with open(os.path.join(out, "findings.json"), "w") as f:
            json.dump(findings_out, f, indent=2)

        # 6. Visualization PNG
        png_path = save_visualization(name, ct_vol, heatmap, binary_mask, out)
        console.print(f"\n[bold green]✓ Saved outputs → {out}[/bold green]")
        console.print(f"  report.txt, embedding.npz, localization_mask.nii.gz")
        console.print(f"  heatmap.nii.gz, findings.json, ct_analysis.png")

        return "done"

    except Exception as e:
        import traceback
        with open(os.path.join(out, "error.txt"), "w") as f:
            f.write(traceback.format_exc())
        print(f"  ✗ Error: {e}")
        return f"error: {e}"

In [ ]:
import time

models = {
    "embed"   : model_embed,
    "pheno"   : model_pheno,
    "fiveyear": model_fiveyear,
    "report"  : model_report,
}

total     = len(matched)
done      = 0
skipped   = 0
errors    = []
log_path  = os.path.join(RESULTS_DIR, "run_log.txt")
start     = time.time()

print(f"Starting pipeline — {total} volumes → {RESULTS_DIR}")

with open(log_path, "a") as log:
    for i, sample in enumerate(matched):
        status = process_volume(sample, models, ct_index,
                                phenotypes_df, RESULTS_DIR)
        if   status == "done"   : done    += 1
        elif status == "skipped": skipped += 1
        else                    : errors.append({"name": sample["name"],
                                                 "error": status})
        if (i+1) % 10 == 0 or (i+1) == total:
            elapsed   = time.time() - start
            per_vol   = elapsed / (i+1)
            remaining = per_vol * (total - i - 1)
            print(f"  [{i+1:4d}/{total}]  done={done}  "
                  f"skip={skipped}  err={len(errors)}  "
                  f"ETA={remaining/60:.1f}min")
        log.write(f"[{i+1}/{total}] {sample['name']} → {status}\n")

print(f"\nFINISHED — processed={done}  skipped={skipped}  errors={len(errors)}")
if errors:
    with open(os.path.join(RESULTS_DIR, "errors.json"), "w") as f:
        json.dump(errors, f, indent=2)